In [ ]:
import os
import subprocess
import numpy as np
from Bio.PDB.PDBParser import PDBParser
import warnings
import yaml
import glob
from rdkit import Chem
from rdkit.Chem.rdMolAlign import CalcRMS
from easydict import EasyDict
import json
import re
import csv
import pandas as pd
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pocket_path = './data_crossdocked/test.yaml'           # './data_crossdocked/test.yaml'
ori_vina_path = '/home/nic/Code/HGNN-GPT/GPT-last-new-2/crossdocked/dock_result2/pocket_vina.csv'
json_file4='./dock_file_save/crossdocked/2025_01_12_16_1743065493/dock_result/dock_dict.json'
json_file3='./dock_file_save/crossdocked/2025_01_05_20_1741635931/dock_result/dock_dict.json'

smiles_yaml4='./save/pre/crossdocked/char/all/2025_01_12_16/sample_300_30_True_1_1_1743065493/1ai4_A_rec_1ai5_mnp_lig_tt_docked_0_pocket10_sampled_temp1.yaml'
smiles_yaml3='./save/pre/crossdocked/char/hgnn/2025_01_05_20/sample_300_30_True_1_1_1741635931/1ai4_A_rec_1ai5_mnp_lig_tt_docked_0_pocket10_sampled_temp1.yaml'

In [ ]:
with open(pocket_path, 'r') as f:
    pocket_dict = yaml.full_load(f)
pocket_names=list(pocket_dict.keys())


ori_vina = {}
with open(ori_vina_path, 'r') as file:
    csv_reader = csv.DictReader(file)
    for row in csv_reader:
        ligand_name = row['pocket_name']
        affinity = float(row['affinity'])
        ori_vina[ligand_name] = affinity

with open(json_file3, 'r') as f:
    dock_data3 = json.load(f)

with open(json_file4, 'r') as f:
    dock_data4 = json.load(f)

with open(smiles_yaml3, 'r') as f:
    mol_dict3 = yaml.full_load(f)
mol_num3=list(mol_dict3.values())

with open(smiles_yaml4, 'r') as f:
    mol_dict4 = yaml.full_load(f)
mol_num4=list(mol_dict4.values())

In [ ]:
print(mol_num3)
print(len(mol_num3))
print(mol_num4)
print(len(mol_num4))

In [ ]:
def get_all_affinity(one_pocket_name,dock_data):
    pass
    affinity_values = {}
    for key, values in dock_data.items():
        for record in values:
            if record.get('mode_id') == 0:
                affinity_values[key] = record.get('affinity', None)
                break
    
    pocket_dock_values={}
    for key,value in affinity_values.items():
        pocket_name = "_".join(key.split("_")[:-1])
        if pocket_name not in pocket_dock_values:
            pocket_dock_values[pocket_name]=[]
        pocket_dock_values[pocket_name].append((key,value))

    one_pocket_affinity = pocket_dock_values[one_pocket_name]

    return [value for _, value in one_pocket_affinity]

In [ ]:
one_pocket_name='1ai4_A_rec_1ai5_mnp_lig_tt_docked_0_pocket10'
one_pocket_name_path='PAC_ECOLX_27_846_0/1ai4_A_rec_1ai5_mnp_lig_tt_docked_0_pocket10.pdb'

In [ ]:
unique_affinity_3=get_all_affinity(one_pocket_name,dock_data3)
print(unique_affinity_3)
print(len(unique_affinity_3))

unique_affinity_4=get_all_affinity(one_pocket_name,dock_data4)
print(unique_affinity_4)
print(len(unique_affinity_4))

In [ ]:
complete_data3 = []
for data, count in zip(unique_affinity_3, mol_num3):
    complete_data3.extend([data] * count)

print(len(complete_data3))
print(complete_data3)

In [ ]:
complete_data4 = []
for data, count in zip(unique_affinity_4, mol_num4):
    complete_data4.extend([data] * count)

print(len(complete_data4))
print(complete_data4)

In [ ]:
df = pd.DataFrame(complete_data3, columns=['PHy2Mol'])
palette = sns.color_palette(['#a1dab4'])

fig, ax = plt.subplots(figsize=(5, 5))
ax = sns.boxplot( data=df,width=0.2,showmeans=True,palette=palette,meanprops={
                                                  "markerfacecolor":"white",
                                                  "markeredgecolor":"black",
                                                  "markersize":"10"})
ax.set_ylabel('vina  score',fontsize=12)
ax.set_xticklabels(['PHy2Mol'])
ax.set_xlim(-0.3, 0.3)
means = df['PHy2Mol'].mean()
ax.text(0.12, means+0.1, f'mean={means:.2f}', ha='left', va='center', fontsize=10, color='black')

median_value = df['PHy2Mol'].median()
ax.text(0.12, median_value-0.1, f'median={median_value:.2f}', ha='left', va='center', fontsize=10, color='black')

plt.tight_layout() 
plt.savefig('./figure/3_vina_score.png',dpi=500)
plt.show()

In [ ]:
df = pd.concat([pd.Series(complete_data3), pd.Series(complete_data4)], axis=1)
df.columns = ['PHy2Mol', 'MB2Mol']
fig, ax = plt.subplots(figsize=(8, 7))
# palette = {"PHy2Mol": "#3ABF99", "MB2Mol": "#ED8D5A"}
palette = {"PHy2Mol": "#a1dab4", "MB2Mol": "#5dbfe9"}
# palette = sns.color_palette(['#a1dab4', '#5dbfe9'])
ax = sns.boxplot( data=df,width=0.3,showmeans=True,palette=palette,meanprops={
                                                  "markerfacecolor":"white",
                                                  "markeredgecolor":"black",
                                                  "markersize":"8"})
ax.set_ylabel('vina  score',fontsize=12)
ax.set_xticklabels(['PHy2Mol', 'MB2Mol'])
# sns.boxplot( x=df,width=0.3,palette=my_pal)


means = df.mean()
for i, (col, mean) in enumerate(means.items()):
    ax.text(i + 0.18, mean + 0.1, f'mean={mean:.2f}', ha='left', va='center', fontsize=10, color='black')

medians = df.median()  
for i, (col, median) in enumerate(medians.items()):
    ax.text(i + 0.18, median - 0.1, f'median={median:.2f}', ha='left', va='center', fontsize=10, color='black')
plt.savefig('./figure/3-4_vina_score.png',dpi=500)
plt.show()
